<a href="https://colab.research.google.com/github/tsgebre/Flood_Physics_Guided_DL/blob/main/experiment_ems_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agent Role: Experiment Evidence & Visualization Engineer**

You are an expert ML research engineer specializing in **experiment reproducibility, evaluation pipelines, and publication-quality visualization**.

Your mission is to **translate evidence requirements (figures, tables, comparisons) into stand-alone scripts** that generate the required results and visuals from existing experiment artifacts.

The notebook contains:

* experiment scripts
* training logs
* example metrics

However:

* **experiments and checkpoints exist on a local machine**
* the scripts you propose will be **executed locally**

Your responsibility is to **design the minimal scripts required to generate the evidences recommended by the previous agent**.

Do **not redesign experiments** unless strictly necessary.

---

# **Primary Objectives**

1. Map evidence requirements to reproducible evaluation workflows.
2. Design **stand-alone scripts** that:

   * load checkpoints
   * run inference or evaluation
   * compute metrics
   * export results
3. Generate **publication-ready figures and tables**.
4. Ensure outputs are easily reproducible and compatible with journal-quality presentation.

---

# **Expected Inputs**

The agent may receive:

* Evidence plan from the previous agent
* Notebook containing experiment scripts and metrics
* Dataset descriptions or loaders
* Partial results or logs
* Model checkpoints (stored locally)

---

# **Input Validation**

Before analysis verify the availability of:

* experiment scripts
* checkpoint paths
* dataset loaders
* metric definitions

If critical components are missing, list them and suggest minimal additions required.

---

# **Operational Framework**

## Phase 1 — Evidence Mapping

For each evidence item determine:

* dataset required
* checkpoint required
* evaluation metric
* expected output artifact

Identify whether the evidence requires:

* inference runs
* metric computation
* aggregation of results
* visualization generation

---

## Phase 2 — Execution Script Design

Design **stand-alone scripts** that can be executed locally.

Scripts may include:

* inference runners
* evaluation pipelines
* metric aggregation utilities
* visualization generators

Each script should specify:

* required inputs
* expected outputs
* execution steps

Prefer **minimal additions** to existing code.

Avoid retraining models unless necessary.

---

## Phase 3 — Result Logging & Artifact Structure

Define reproducible outputs using structured formats:

Recommended artifacts:

```
metrics.csv
results_summary.csv
predictions.npy
metadata.json
```

Each artifact should record:

* experiment_id
* checkpoint_used
* dataset_split
* metric_definition

---

## Phase 4 — Publication-Quality Visualization Design

For each required visual element, specify:

* figure type (plot, chart, table)
* data source
* variables shown
* layout structure

Ensure figures are **publication-ready**:

Guidelines:

* font sizes appropriate for journal figures
* clear axis labeling and legends
* colorblind-safe color palettes
* consistent styling across figures
* export formats suitable for papers (`PDF`, `SVG`, high-resolution `PNG`)

Tables should be structured for **direct inclusion in manuscripts**.

---

# **Output Format**

### 1. Evidence Implementation Summary

Mapping between evidence items and required experiment outputs.

### 2. Missing Components

List scripts or utilities needed to generate results.

### 3. Proposed Execution Scripts

For each script describe:

* purpose
* inputs
* outputs
* execution steps

(No full code.)

### 4. Experiment Output Structure

Example:

```
artifacts/
   experiment_01/
      metrics.csv
      predictions.npy
      metadata.json
```

### 5. Visualization Specification

Describe the figures/tables to be generated and how they should present the results.

### 6. Evidence Readiness Check

Confirm whether the outputs will support the required figures and tables.

---

# **Escalation Condition**

Escalate as **“Evidence Generation Blocked”** if:

* checkpoints are unavailable
* datasets cannot be accessed
* required metrics cannot be computed

Provide minimal corrective actions.

---

# **Behavioral Constraints**

* Prefer **evaluation and inference using existing checkpoints**.
* Avoid generic ML advice.
* Focus strictly on **scripts and workflows needed to generate evidence**.
* Ensure figures are **clear, consistent, and publication-ready**.

---

# **Tone**

Technical, concise, and implementation-focused.
Assume an **ML research audience preparing reproducible experiments for publication**.

## Scripts

```
# main.py

import torch
import numpy as np
import pandas as pd
import argparse
import os
import random

import src.dataset as dataset
import src.models as models
import src.train as train
import src.utils as utils
from torch.utils.data import DataLoader


# ------------------------
# Reproducibility
# ------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ------------------------
# Experiment runner
# ------------------------
def run_experiment(model_type, hidden_dim, lr, physics_weight,
                   use_physics, device, loaders, scaler_d, seed):

    ModelClass = (
        models.PhysicsInformedLSTM if model_type == 'LSTM'
        else models.PhysicsInformedGRU
    )

    print(f"\n--- {model_type} | {'Physics' if use_physics else 'Control'} ---")

    init_rain, init_flood = loaders['thresholds']

    model = ModelClass(
        2, hidden_dim, 2, 1,   # precip + missing flag
        init_rain,
        init_flood,
        init_weight=physics_weight
    ).to(device)

    model = train.train_model(
        model,
        loaders['train'],
        loaders['val'],
        lr=lr,
        use_physics=use_physics,
        epochs=50,
        patience=10,
        device=device
    )

    preds, targets = utils.evaluate_model(
        model,
        loaders['test'],
        scaler_d,
        device=device
    )

    preds = np.asarray(preds)
    targets = np.asarray(targets)

    mae = np.mean(np.abs(targets - preds))
    rmse = np.sqrt(np.mean((targets - preds) ** 2))
    nse = utils.calculate_nse(targets.flatten(), preds.flatten())

    return {"MAE": mae, "RMSE": rmse, "NSE": nse}


# ------------------------
# Basins & scenarios
# ------------------------
TARGET_BASINS = [
    '01013500','02046000','02215100','03338780','05507600',
    '06885500','08165300','09066300','11532500','13331500'
]

OUTAGE_SCENARIOS = [0.0, 0.2, 0.5, 0.8]


# ------------------------
# Main experiment loop
# ------------------------
def run_multibasin_experiment(args, device):

    print(f"\n=== Multi-Basin Evaluation ({len(TARGET_BASINS)} basins) ===")

    results = []
    base_dir = os.path.join(os.path.dirname(__file__), "data")

    # hyperparameters
    if args.model == 'LSTM':
        hidden_dim, lr, physics_weight = 128, 0.0018, 0.10
    else:
        hidden_dim, lr, physics_weight = 64, 0.0011, 0.14

    for basin in TARGET_BASINS:
        for outage_ratio in OUTAGE_SCENARIOS:

            print(f"\n>>> Basin: {basin} | Missing={int(outage_ratio*100)}%")

            try:
                # ------------------------
                # Load data
                # ------------------------
                streamflow_path = os.path.join(base_dir, f"{basin}_streamflow_qc.txt")
                forcing_path = os.path.join(base_dir, f"{basin}_lump_cida_forcing_leap.txt")

                train_df, val_df, test_df, scaler_p, scaler_d, init_rain, init_flood = \
                    dataset.load_and_preprocess_data(streamflow_path, forcing_path)

                # ------------------------
                # Apply controlled test masking
                # ------------------------
                test_df_masked = test_df.copy()

                test_df_masked["discharge_raw"] = test_df["discharge"].values

                if outage_ratio > 0:
                    test_df_masked["discharge_raw"] = dataset.simulate_block_sensor_failure(
                        test_df["discharge"].values,
                        missing_ratio=outage_ratio,
                        block_length=14
                    )

                test_df_masked["missing_flag"] = np.isnan(test_df_masked["discharge_raw"]).astype(float)
                test_df_masked["discharge"] = test_df_masked["discharge_raw"]

                # ------------------------
                # Interpolation baseline
                # ------------------------
                if outage_ratio == 0:
                    interp_nse = np.nan
                else:
                    true_series = test_df["discharge"].values
                    masked_series = test_df_masked["discharge_raw"].copy()

                    interp_pred = dataset.linear_interpolation_baseline(masked_series)

                    missing = np.isnan(masked_series)

                    if np.any(missing):
                        interp_nse = utils.calculate_nse(
                            true_series[missing],
                            interp_pred[missing]
                        )
                    else:
                        interp_nse = np.nan

                # ------------------------
                # DataLoaders
                # ------------------------
                g_train = torch.Generator().manual_seed(args.seed)
                g_val = torch.Generator().manual_seed(args.seed + 1)
                g_test = torch.Generator().manual_seed(args.seed + 2)

                train_dataset = dataset.CamelsDataset(
                    train_df, 30, is_train=True, outage_ratio=0.2
                )

                val_dataset = dataset.CamelsDataset(
                    val_df, 30, is_train=False, outage_ratio=0.0
                )

                test_dataset = dataset.CamelsDataset(
                    test_df_masked, 30, is_train=False, outage_ratio=0.0
                )

                loaders = {
                    "train": DataLoader(
                        train_dataset,
                        batch_size=32,
                        shuffle=True,
                        worker_init_fn=seed_worker,
                        generator=g_train
                    ),
                    "val": DataLoader(
                        val_dataset,
                        batch_size=32,
                        shuffle=False,
                        worker_init_fn=seed_worker,
                        generator=g_val
                    ),
                    "test": DataLoader(
                        test_dataset,
                        batch_size=32,
                        shuffle=False,
                        worker_init_fn=seed_worker,
                        generator=g_test
                    ),
                    "thresholds": (init_rain, init_flood)
                }

                # ------------------------
                # Run models
                # ------------------------
                res_ctrl = run_experiment(
                    args.model, hidden_dim, lr, physics_weight,
                    False, device, loaders, scaler_d, args.seed
                )

                res_pi = run_experiment(
                    args.model, hidden_dim, lr, physics_weight,
                    True, device, loaders, scaler_d, args.seed
                )

                # ------------------------
                # Store results
                # ------------------------
                results.append({
                    "Basin": basin,
                    "MissingRatio": outage_ratio,

                    "Control_NSE": res_ctrl["NSE"],
                    "PI_NSE": res_pi["NSE"],
                    "Interp_NSE": interp_nse,

                    "Control_MAE": res_ctrl["MAE"],
                    "PI_MAE": res_pi["MAE"],

                    "Control_RMSE": res_ctrl["RMSE"],
                    "PI_RMSE": res_pi["RMSE"],

                    "Physics_Gain": res_pi["NSE"] - res_ctrl["NSE"]
                })

                print(f"✔ Completed {basin}")

            except Exception as e:
                print(f"⚠ Skipping {basin}: {e}")


    # ------------------------
    # Save + summary
    # ------------------------
    df_out = pd.DataFrame(results)
    df_out.to_csv("multibasin_results.csv", index=False)

    print("\n==============================")
    print("AVERAGE NSE BY OUTAGE LEVEL")
    print("==============================")

    if len(df_out) > 0:
        print(
            df_out.groupby("MissingRatio")[
                ["Control_NSE", "PI_NSE", "Interp_NSE", "Physics_Gain"]
            ].mean().round(4)
        )
    else:
        print("No results to summarize.")


# ------------------------
# Main entry
# ------------------------
if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument('--model', choices=['LSTM', 'GRU'], default='LSTM')
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_multibasin_experiment(args, device)

# train.py

import torch
import torch.optim as optim
import torch.nn.functional as F
import copy


def train_model(model, train_loader, val_loader,
                lr, use_physics=False,
                epochs=50, patience=10,
                device='cpu'):

    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    print(f"\nTraining {model.__class__.__name__}")
    print(f"Physics: {use_physics}")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)

            optimizer.zero_grad()
            x = x.clone()
            out = model(x)

            # SAFE aggregation (prevents mismatch across models)
            y_pred = out[:, -1, :]

            # loss = F.mse_loss(y_pred, y)
            # y = batch['y'].to(device)

            loss = F.mse_loss(out, y)

            if use_physics:
                q = out
                r = x[:, :, 0:1]

                dq = q[:, 1:, :] - q[:, :-1, :]
                dr = r[:, 1:, :] - r[:, :-1, :]

                monotonic_penalty = torch.mean(torch.relu(-dq * dr))

                t_rain = torch.relu(model.thresh_rain)
                t_flood = torch.relu(model.thresh_flood)

                flood_penalty = F.relu(t_flood - q).mean()

                loss = loss + (
                    torch.relu(model.w_monotonicity) * monotonic_penalty +
                    torch.relu(model.w_threshold) * flood_penalty
                    )

                reg = 1e-4 * (
                    model.thresh_rain.pow(2) +
                    model.thresh_flood.pow(2)
                )
                loss = loss + reg

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # validation
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(device)
                y = batch['y'].to(device)

                out = model(x)
                y_pred = out[:, -1, :]

                # val_loss += F.mse_loss(y_pred, y).item()
                val_loss += F.mse_loss(out, y).item()

        val_loss /= len(val_loader)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

        if epoch % 2 == 0:
            print(f"Epoch {epoch} | Val Loss: {val_loss:.5f}")

    model.load_state_dict(best_state)

    print(f"Best Val Loss: {best_loss:.5f}")
    return model
  
# dataset.py

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler

class CamelsDataset(Dataset):
    def __init__(self, df, seq_len=30, is_train=False, outage_ratio=0.0):
        self.seq_len = seq_len
        self.is_train = is_train
        self.outage_ratio = outage_ratio

        # ALWAYS 2 FEATURES ONLY
        features = df[['precip_norm', 'missing_flag']].copy()

        self.x = torch.tensor(features.values, dtype=torch.float32)
        y = df['discharge_norm'].values
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

        # Precompute deterministic mask for test stress scenarios
        if not is_train and outage_ratio > 0:
            self.masked_x = self._apply_block_mask(self.x.clone(), outage_ratio)
        else:
            self.masked_x = self.x

    def __len__(self):
        return len(self.x) - self.seq_len

    def _apply_block_mask(self, x, ratio):
        seq_len = x.shape[0]
        n_missing = int(seq_len * ratio)
        block_len = 14
        n_blocks = max(1, n_missing // block_len)

        x = x.clone()

        for _ in range(n_blocks):
            start = np.random.randint(0, seq_len - block_len)
            end = start + block_len

            x[start:end, 0] = 0.0   # precip removed
            x[start:end, 1] = 1.0   # missing flag

        return x

    def __getitem__(self, idx):
        if self.is_train:
            x_seq = self.x[idx: idx + self.seq_len].clone()
        else:
            x_seq = self.masked_x[idx: idx + self.seq_len].clone()

        # y_target = self.y[idx + self.seq_len].clone()

        y_seq = self.y[idx: idx + self.seq_len]
        return {'x': x_seq, 'y': y_seq}

        # return {
        #     'x': x_seq,
        #     'y': y_target
        # }


def load_and_preprocess_data(streamflow_path, forcing_path):
    df_flow = pd.read_csv(
        streamflow_path, sep=r'\s+', header=None,
        names=['basin', 'year', 'month', 'day', 'discharge', 'qc']
    )
    df_flow['date'] = pd.to_datetime(df_flow[['year', 'month', 'day']])

    df_forcing = pd.read_csv(forcing_path, skiprows=3, sep=r'\s+')
    df_forcing.rename(columns={'Mnth': 'month', 'Year': 'year', 'Day': 'day'}, inplace=True)
    df_forcing['date'] = pd.to_datetime(df_forcing[['year', 'month', 'day']])

    precip_col = [c for c in df_forcing.columns if 'prcp' in c.lower()][0]

    df = pd.merge(
        df_flow[['date', 'discharge']],
        df_forcing[['date', precip_col]],
        on='date'
    )

    df.rename(columns={precip_col: 'precipitation'}, inplace=True)

    df = df[(df['date'] >= '1980-01-01') & (df['date'] <= '1999-12-31')].copy()

    df['discharge'] = df['discharge'].replace(-999, np.nan)
    df['discharge'] = df['discharge'].interpolate()
    df.dropna(inplace=True)

    split1 = int(len(df) * 0.7)
    split2 = int(len(df) * 0.85)

    train_df = df.iloc[:split1].copy()
    val_df = df.iloc[split1:split2].copy()
    test_df = df.iloc[split2:].copy()

    scaler_p = StandardScaler()
    scaler_d = StandardScaler()

    scaler_p.fit(train_df[['precipitation']])
    scaler_d.fit(train_df[['discharge']])

    for d in [train_df, val_df, test_df]:
        d['precip_norm'] = scaler_p.transform(d[['precipitation']])
        d['discharge_norm'] = scaler_d.transform(d[['discharge']])
        d['missing_flag'] = 0.0

    init_rain = train_df['precip_norm'].quantile(0.95)
    init_flood = train_df['discharge_norm'].quantile(0.95)

    return train_df, val_df, test_df, scaler_p, scaler_d, init_rain, init_flood

def simulate_block_sensor_failure(discharge_series, missing_ratio=0.8, block_length=14):
    """
    Replaces random masking with contiguous block masking (e.g., 14-day outages)
    to simulate real-world sensor washouts during floods.
    """
    # Ensure we are working with a pandas Series for easy NaN checking/assignment
    is_numpy = isinstance(discharge_series, np.ndarray)
    if is_numpy:
        masked_series = pd.Series(discharge_series.flatten())
    else:
        masked_series = discharge_series.copy()
        
    n_total = len(masked_series)
    n_missing_target = int(n_total * missing_ratio)
    n_blocks = n_missing_target // block_length

    available_indices = np.arange(n_total - block_length)
    np.random.shuffle(available_indices)

    blocks_applied = 0
    for start_idx in available_indices:
        if blocks_applied >= n_blocks:
            break
        # Ensure we don't overlap with already masked NaNs
        if not np.isnan(masked_series.iloc[start_idx:start_idx+block_length]).any():
            masked_series.iloc[start_idx:start_idx+block_length] = np.nan
            blocks_applied += 1

    if is_numpy:
        return masked_series.values.reshape(discharge_series.shape)
    return masked_series

def linear_interpolation_baseline(masked_series):
    """
    Statistical imputation baseline to compare against PI-LSTM.
    """
    if isinstance(masked_series, pd.Series) or isinstance(masked_series, pd.DataFrame):
        return masked_series.interpolate(method='linear', limit_direction='both')
    else:
        # Fallback for numpy arrays
        s = pd.Series(masked_series.flatten())
        return s.interpolate(method='linear', limit_direction='both').values


  
# models.py

import torch
import torch.nn as nn

class PhysicsInformedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, init_rain, init_flood, init_weight=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Learnable Parameters
        # init_weight allows setting initial physics weights (0.0 for Control)
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))

        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        # Initialize hidden state on the same device as input x
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out)


# utils.py

import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def calculate_nse(y_true, y_pred):
    """Calculates the Nash-Sutcliffe Efficiency."""
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))

def evaluate_model(model, loader, scaler, device='cpu'):
    """Sequence-to-sequence evaluation."""
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)

            outputs = model(x)

            preds.append(outputs.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)

    # Flatten (batch, seq_len, 1) -> (N, 1)
    preds = preds.reshape(-1, 1)
    targets = targets.reshape(-1, 1)

    preds_orig = scaler.inverse_transform(preds)
    targets_orig = scaler.inverse_transform(targets)

    return preds_orig, targets_orig

def evaluate_stress_test(
    model,
    loader,
    scaler,
    mask_ratio=0.5,
    block_length=14,
    device='cpu'
):
    """
    Evaluates model under realistic sensor outages.
    """
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for batch in loader:

            x = batch['x'].to(device)
            y = batch['y'].to(device)

            x_in = x.clone()

            if mask_ratio > 0:

                batch_size = x.shape[0]
                seq_len = x.shape[1]

                for b in range(batch_size):

                    n_missing = int(seq_len * mask_ratio)
                    n_blocks = max(1, n_missing // block_length)

                    for _ in range(n_blocks):

                        start = np.random.randint(
                            0,
                            max(1, seq_len - block_length)
                        )

                        end = min(start + block_length, seq_len)

                        x_in[b, start:end, 1] = 0.0

            outputs = model(x_in)

            preds.append(outputs.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)

    preds = preds.reshape(-1, 1)
    targets = targets.reshape(-1, 1)

    preds_orig = scaler.inverse_transform(preds)
    targets_orig = scaler.inverse_transform(targets)

    return preds_orig, targets_orig
